### Data downloading

Follow the [instructions in README.md](https://github.com/allenai/spoc-robot-training?tab=readme-ov-file#-downloading-assets-annotations-and-houses-) to get all data downloaded (annotations, assets, and houses).

In [ ]:
OBJAVERSE_DATA_DIR = "/path/to/objaverse_assets"  # use here the path where assets and annotations were downloaded to
OBJAVERSE_HOUSES_DIR = "/path/to/objaverse_houses/houses_2023_07_28"  # use here the path where houses were downloaded to

%set_env OBJAVERSE_DATA_DIR=$OBJAVERSE_DATA_DIR
%set_env OBJAVERSE_HOUSES_DIR=$OBJAVERSE_HOUSES_DIR

### Using annotations

This example shows how to load annotations and lists the contents for a random asset UID.

In [ ]:
from utils.objaverse_annotation import get_objaverse_annotations
import random
import json

# Gets the annotation dict
annotation = get_objaverse_annotations()

print(f"{len(annotation)} entries")
print(f"Sample:\n{json.dumps(annotation[random.choice(list(annotation.keys()))], indent=2)}")

Let's find the frequency of (available) GPT-generated descriptions and list the most commonly appearing ones.

In [ ]:
from collections import Counter
counts = Counter()
for entry in annotation.values():
    counts[entry["description_auto"]] += 1
print(f"Most common: {counts.most_common(5)}")

### Loading houses and required assets

First, let's see how to load house data. For example, load the first house from the `val` partition.

In [ ]:
import os

import prior

from utils.constants.objaverse_data_dirs import OBJAVERSE_HOUSES_DIR

houses = prior.load_dataset(
    dataset="spoc-data",
    entity="spoc-robot",
    revision="local-objaverse-procthor-houses",
    path_to_splits=None,
    split_to_path={
        k: os.path.join(OBJAVERSE_HOUSES_DIR, f"{k}.jsonl.gz")
        for k in ["val"]
    },
    max_houses_per_split=2,
)['val']

print(f"{len(houses)} loaded val houses")
print(f"Sample room description:\n{json.dumps(houses[0]['rooms'][0], indent=2)}")

Now, let's load the house (and all required assets) as a THOR scene. The agent is the Stretch Robot with two RGB cameras (navigation and manipulation) used in our paper.

In [ ]:
print(houses)

In [ ]:
from PIL import Image

from environment.stretch_controller import StretchController
from utils.constants.stretch_initialization_utils import STRETCH_ENV_ARGS

c = StretchController(**STRETCH_ENV_ARGS)
event = c.reset(houses[0])
# print(houses[0])

# print(event)
print("\nnav")
display(Image.fromarray(c.navigation_camera))
# print("\nmanip")
# display(Image.fromarray(c.manipulation_camera))

We can also easily extract available descriptions for objects in the environment. In this example, let's consider all objects visible in each camera up to a maximum distance of 5 meters:

In [ ]:
print("Navigation camera:")
for it, obj in enumerate(c.get_visible_objects("nav", maximum_distance=5)):
    print(it, c.get_object(obj)["description"])

# print("\nManipulation camera:")
# for it, obj in enumerate(c.get_visible_objects("manip", maximum_distance=5)):
#     print(it, c.get_object(obj)["description"])

c.stop()